In [ ]:
# Install required packages
!pip install pycryptodome ecdsa bitcoin base58 tqdm pybloom-live torch

import hashlib
import binascii
from Crypto.Hash import RIPEMD160
import ecdsa
import base58
import random
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
import concurrent.futures
import threading
import os
from google.colab import files
from pybloom_live import ScalableBloomFilter
import torch

# Check if GPU is available and set it up
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set up display and threading for tracking global bests
print_lock = threading.Lock()
global_bests = {}

# Function to compute Bitcoin address from private key
def private_key_to_address(private_key_hex, compressed=False):
    # Convert hex private key to bytes
    private_key_bytes = bytes.fromhex(private_key_hex.zfill(64))

    # Get public key from private key
    signing_key = ecdsa.SigningKey.from_string(private_key_bytes, curve=ecdsa.SECP256k1)
    verifying_key = signing_key.get_verifying_key()

    # Get public key in bytes form
    if compressed:
        if verifying_key.pubkey.point.y() % 2 == 0:
            public_key_bytes = b'\x02' + verifying_key.pubkey.point.x().to_bytes(32, 'big')
        else:
            public_key_bytes = b'\x03' + verifying_key.pubkey.point.x().to_bytes(32, 'big')
    else:
        public_key_bytes = b'\x04' + verifying_key.pubkey.point.x().to_bytes(32, 'big') + verifying_key.pubkey.point.y().to_bytes(32, 'big')

    # Compute SHA256 hash of public key
    sha256_hash = hashlib.sha256(public_key_bytes).digest()

    # Compute RIPEMD160 hash of SHA256 hash
    ripemd160_hash = RIPEMD160.new(sha256_hash).digest()

    # Add version byte (0x00 for mainnet, 0x6f for testnet)
    versioned_hash = b'\x00' + ripemd160_hash

    # Compute checksum (first 4 bytes of double SHA256 of versioned hash)
    checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]

    # Combine versioned hash and checksum
    binary_address = versioned_hash + checksum

    # Convert to Base58 encoding
    address = base58.b58encode(binary_address).decode('utf-8')

    return address, ripemd160_hash

# GPU-accelerated Hamming distance calculation
def batch_hamming_distance_gpu(keys_batch, hashes_batch, batch_size=1000):
    """
    Calculate Hamming distances for a batch of keys against their hashes using GPU

    Args:
        keys_batch: List of key byte arrays (each 20 bytes)
        hashes_batch: List of hash160 byte arrays (each 20 bytes)
        batch_size: Number of calculations to perform in each GPU batch

    Returns:
        List of Hamming distances
    """
    # Convert byte arrays to bit arrays
    keys_bits = []
    hashes_bits = []

    for i in range(len(keys_batch)):
        # Convert each byte to 8 bits
        key_bits = []
        hash_bits = []

        for j in range(20):  # 20 bytes
            key_byte = keys_batch[i][j] if j < len(keys_batch[i]) else 0
            hash_byte = hashes_batch[i][j] if j < len(hashes_batch[i]) else 0

            for k in range(8):  # 8 bits per byte
                key_bits.append((key_byte >> k) & 1)
                hash_bits.append((hash_byte >> k) & 1)

        keys_bits.append(key_bits)
        hashes_bits.append(hash_bits)

    # Convert to PyTorch tensors
    keys_tensor = torch.tensor(keys_bits, dtype=torch.uint8, device=device)
    hashes_tensor = torch.tensor(hashes_bits, dtype=torch.uint8, device=device)

    # XOR the tensors to get differences
    diff_tensor = torch.bitwise_xor(keys_tensor, hashes_tensor)

    # Sum along bit dimension to get Hamming distances
    distances = torch.sum(diff_tensor, dim=1).cpu().numpy()

    return distances

# Batch key generation for GPU processing
def generate_keys_batch(start_int, count):
    """Generate a batch of private keys from integers"""
    keys = []
    for i in range(count):
        key_int = start_int + i
        key_hex = format(key_int, 'x').zfill(64)
        key_bytes = bytes.fromhex(key_hex)[-20:]  # Last 20 bytes for comparison with hash160
        keys.append(key_bytes)
    return keys

# GPU-accelerated pre-screening
def gpu_prescreen_keys(private_key_int, batch_size, count, threshold=100):
    """
    Use GPU to quickly screen keys that might have low Hamming distance

    This is a simplified version that uses a heuristic based on patterns
    in the key bytes that might correlate with low Hamming distance.
    """
    # Generate batch of sequential keys
    key_ints = torch.arange(private_key_int, private_key_int + count, device=device)

    # Extract patterns from keys (simplified approach)
    # This is where a machine learning model could be used to predict promising keys
    # For now, we'll use a simple heuristic based on bit patterns

    # Convert to 64-bit representation for bit operations
    key_bits = torch.zeros((count, 64), dtype=torch.uint8, device=device)
    for i in range(64):
        key_bits[:, i] = (key_ints >> i) & 1

    # Calculate a score based on bit patterns
    # This is a simplified heuristic - would need to be tuned
    scores = torch.sum(key_bits, dim=1)  # Count ones

    # Select keys with promising scores
    # Lower scores might indicate potential for lower Hamming distance
    promising_indices = torch.where(scores < threshold)[0].cpu().numpy()
    promising_keys = [int(key_ints[i].item()) for i in promising_indices]

    return promising_keys

# Function to calculate Hamming distance between private key and RIPEMD160 hash
def calculate_key_hash_distance(private_key_hex, compressed=False):
    # Get private key in bytes
    if len(private_key_hex) < 64:
        private_key_hex = private_key_hex.zfill(64)
    private_key_bytes = bytes.fromhex(private_key_hex)

    # Ensure private key is 32 bytes (hash160 is 20 bytes)
    private_key_bytes = private_key_bytes[-20:]  # Use last 20 bytes for comparison

    # Get Bitcoin address and RIPEMD160 hash
    address, hash160 = private_key_to_address(private_key_hex, compressed)

    # Calculate Hamming distance on CPU (for individual calculations)
    distance = 0
    for b1, b2 in zip(private_key_bytes, hash160):
        xor = b1 ^ b2
        # Count bits in xor result
        while xor:
            distance += xor & 1
            xor >>= 1

    # Calculate percentage
    percentage = 100 - (distance * 0.625)

    return address, private_key_hex, distance, percentage, hash160

# GPU-accelerated batch processing
def process_keys_gpu(target_addresses, start_int, batch_size, threshold=36):
    """Process a batch of keys using GPU acceleration for screening and Hamming distance calculation"""
    results = []

    # 1. Generate candidate keys (integers)
    candidate_ints = list(range(start_int, start_int + batch_size))

    # 2. Pre-screen using GPU heuristics
    promising_keys = gpu_prescreen_keys(start_int, batch_size, 100)  # Threshold for pre-screening

    print(f"Pre-screening found {len(promising_keys)} promising keys out of {batch_size}")

    # 3. Process promising keys in smaller GPU batches
    sub_batch_size = 1000  # Process this many at a time on GPU
    for i in range(0, len(promising_keys), sub_batch_size):
        sub_batch = promising_keys[i:i+sub_batch_size]

        # Generate key bytes and calculate addresses, hashes
        key_bytes_list = []
        address_list = []
        hash160_list = []

        for key_int in sub_batch:
            key_hex = format(key_int, 'x').zfill(64)
            key_bytes = bytes.fromhex(key_hex)[-20:]  # Last 20 bytes

            # Calculate address and hash160
            address, hash160 = private_key_to_address(key_hex, False)  # Uncompressed

            key_bytes_list.append(key_bytes)
            address_list.append(address)
            hash160_list.append(hash160)

        # Calculate Hamming distances using GPU
        distances = batch_hamming_distance_gpu(key_bytes_list, hash160_list)

        # Process results
        for j in range(len(sub_batch)):
            key_int = sub_batch[j]
            key_hex = format(key_int, 'x').zfill(64)
            address = address_list[j]
            distance = distances[j]
            percentage = 100 - (distance * 0.625)

            # Check if Hamming distance meets threshold
            if distance <= threshold:
                result = (address, key_hex, int(distance), percentage, 0.0, False, key_int)

                # Check if in target list
                if address in target_addresses:
                    results.append(result)

                # Update global best
                with print_lock:
                    if address not in global_bests or distance < global_bests[address][2]:
                        global_bests[address] = result
                        print(f"NEW BEST for {address}: Key={key_hex}, Distance={distance}, Percentage={percentage:.2f}%, Integer={key_int}")

    return results

# Function to check a batch of keys using multi-threading and GPU acceleration
def check_addresses_batch(target_addresses, private_key_int, start_idx, end_idx, threshold=36, chunk_id=0):
    # Initialize stats
    keys_screened = end_idx - start_idx
    keys_processed = 0

    # Calculate batch size for GPU processing
    gpu_batch_size = 100000
    results = []

    # Process in GPU batches
    for batch_start in range(start_idx, end_idx, gpu_batch_size):
        batch_end = min(batch_start + gpu_batch_size, end_idx)
        batch_results = process_keys_gpu(
            target_addresses,
            private_key_int + batch_start,
            batch_end - batch_start,
            threshold
        )
        results.extend(batch_results)
        keys_processed += len(batch_results)

    # Log progress
    with print_lock:
        print(f"Chunk {chunk_id}: Screened {keys_screened} keys, found {keys_processed} results")

    return results, keys_screened, keys_processed

# Function to load target addresses from previous data
def load_target_addresses_from_data(data_string):
    addresses = []
    lines = data_string.strip().split('\n')
    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 1:
            addresses.append(parts[0])
    return addresses

# Function to load previous results to set initial global bests
def load_previous_results(data_string):
    lines = data_string.strip().split('\n')
    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 6:
            address = parts[0]
            key_hex = parts[1]
            try:
                distance = int(parts[2])
                percentage = float(parts[3])
                entropy = float(parts[4])
                compressed = (parts[5] == 'True')
                # Convert hex key to integer
                current_key = int(key_hex, 16)

                # Store in global bests
                global_bests[address] = (address, key_hex, distance, percentage, entropy, compressed, current_key)
                print(f"Loaded previous best for {address}: Distance={distance}, Percentage={percentage:.2f}%, Integer={current_key}")
            except (ValueError, IndexError) as e:
                print(f"Warning: Couldn't parse line: {line}")

    return global_bests

# Main function to run the search with GPU acceleration
def run_search(target_addresses, start_from_int, num_keys_to_check, threshold=36, num_threads=8, batch_size=1000000):
    print(f"Starting GPU-accelerated search with {num_threads} threads from integer {start_from_int}, checking {num_keys_to_check} keys")
    print(f"Using threshold < {threshold} for Hamming distance")

    all_results = []
    start_time = time.time()
    batch_counter = 0

    # Stats tracking
    total_keys_screened = 0
    total_keys_processed = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        for batch_start in range(0, num_keys_to_check, batch_size):
            batch_counter += 1
            batch_end = min(batch_start + batch_size, num_keys_to_check)

            # Split batch into chunks for each thread
            chunk_size = (batch_end - batch_start) // num_threads
            futures = []

            for i in range(num_threads):
                chunk_start = batch_start + i * chunk_size
                chunk_end = batch_start + (i + 1) * chunk_size if i < num_threads - 1 else batch_end

                # Submit chunk to thread pool
                future = executor.submit(
                    check_addresses_batch,
                    target_addresses,
                    start_from_int,
                    chunk_start,
                    chunk_end,
                    threshold,
                    f"B{batch_counter}-T{i}"
                )
                futures.append(future)

            # Collect results from threads
            batch_results = []
            batch_screened = 0
            batch_processed = 0

            for future in concurrent.futures.as_completed(futures):
                results, keys_screened, keys_processed = future.result()
                batch_results.extend(results)
                batch_screened += keys_screened
                batch_processed += keys_processed

            all_results.extend(batch_results)

            # Update stats
            total_keys_screened += batch_screened
            total_keys_processed += batch_processed

            # Report progress
            elapsed = time.time() - start_time
            keys_checked = batch_end
            keys_per_second = keys_checked / elapsed if elapsed > 0 else 0

            print(f"\n--- PROGRESS REPORT ---")
            print(f"Keys checked: {keys_checked:,}/{num_keys_to_check:,} ({keys_checked/num_keys_to_check*100:.2f}%)")
            print(f"Speed: {keys_per_second:.2f} keys/second, Elapsed: {elapsed:.2f} seconds")
            print(f"Fully processed: {total_keys_processed:,} ({total_keys_processed/total_keys_screened*100:.4f}%)")
            print(f"Current global bests: {len(global_bests)} addresses with improved Hamming distances")

            # Show top 5 best results so far
            print("\n--- TOP 5 BEST RESULTS ---")
            if global_bests:
                sorted_bests = sorted(global_bests.items(), key=lambda x: x[1][2])
                for i, (address, (addr, key_hex, distance, percentage, entropy, compressed, integer)) in enumerate(sorted_bests[:5]):
                    print(f"{i+1}. {addr}: Distance={distance}, Percentage={percentage:.2f}%, Integer={integer}")

            # Save intermediate results
            if batch_counter % 5 == 0:
                save_results_to_csv(all_results, 'intermediate_results.csv')
                save_global_bests_to_csv(global_bests, 'global_bests.csv')

                # Clear accumulated results to save memory
                all_results = []

    # Final save
    save_results_to_csv(all_results, 'final_results.csv')
    save_global_bests_to_csv(global_bests, 'global_bests.csv')

    return all_results

# Function to save results to CSV
def save_results_to_csv(results, filename):
    if not results:
        print(f"No results to save to {filename}")
        return

    df = pd.DataFrame(results, columns=['Address', 'PrivateKeyHex', 'HammingDistance', 'Percentage', 'Entropy', 'Compressed', 'Integer'])
    df.to_csv(filename, index=False)
    print(f"Saved {len(results)} results to {filename}")

# Function to save global bests to CSV
def save_global_bests_to_csv(bests, filename):
    if not bests:
        print(f"No global bests to save to {filename}")
        return

    data = []
    for address, (addr, key_hex, distance, percentage, entropy, compressed, integer) in bests.items():
        data.append([addr, key_hex, distance, percentage, entropy, compressed, integer])

    df = pd.DataFrame(data, columns=['Address', 'PrivateKeyHex', 'HammingDistance', 'Percentage', 'Entropy', 'Compressed', 'Integer'])
    df.to_csv(filename, index=False)
    print(f"Saved {len(bests)} global bests to {filename}")

# Upload function for user to provide the data
def upload_and_start():
    print("Please upload your existing data file (mthfr.txt or similar):")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Using empty target list.")
        target_addresses = []
    else:
        file_name = list(uploaded.keys())[0]
        data_string = uploaded[file_name].decode('utf-8')
        target_addresses = load_target_addresses_from_data(data_string)
        load_previous_results(data_string)

        print(f"Loaded {len(target_addresses)} target addresses.")

    # Get user input for search parameters
    start_from_str = input("Enter starting integer (e.g., 0, 1000000): ")
    start_from_int = int(start_from_str)

    num_keys_str = input("Enter number of keys to check (e.g., 1000000000): ")
    num_keys_to_check = int(num_keys_str)

    threshold_str = input("Enter Hamming distance threshold (default 36): ")
    threshold = int(threshold_str) if threshold_str else 36

    num_threads_str = input("Enter number of threads to use (default 8 for GPU coordination): ")
    num_threads = int(num_threads_str) if num_threads_str else 8

    batch_size_str = input("Enter batch size (default 1,000,000): ")
    batch_size = int(batch_size_str) if batch_size_str else 1000000

    # Run the search
    results = run_search(target_addresses, start_from_int, num_keys_to_check, threshold, num_threads, batch_size)

    # Print final summary
    print("\nSearch completed!")
    print(f"Total matches found: {len(results)}")
    print("Global bests:")
    for address, (addr, key_hex, distance, percentage, entropy, compressed, integer) in sorted(global_bests.items(), key=lambda x: x[1][2]):
        print(f"{addr}: Distance={distance}, Percentage={percentage:.2f}%, Key={key_hex}, Integer={integer}")

    # Download results
    files.download('global_bests.csv')
    files.download('final_results.csv')

# GPU warming up - small initial calculation to initialize CUDA
def warm_up_gpu():
    if torch.cuda.is_available():
        print("Warming up GPU...")
        x = torch.zeros(1000, 1000, device=device)
        y = torch.matmul(x, x)
        torch.cuda.synchronize()
        print("GPU ready!")

# Start with GPU warmup
warm_up_gpu()

# Start the program
upload_and_start()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 115.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvji

Streaming output truncated to the last 5000 lines.
3. 1KNxYaLy5HGZmKGH8NzJzUVhomHg84d2y2: Distance=41, Percentage=74.38%, Integer=132031863
4. 1ZhKkzRLLfpNGNHf8qDDH7uRZtaMue7tR: Distance=41, Percentage=74.38%, Integer=23982329
5. 1khvHY2vNmLbZAwySCvJLs4GzJQfdHmjc: Distance=41, Percentage=74.38%, Integer=51333754
No results to save to intermediate_results.csv
Saved 999 global bests to global_bests.csv
Pre-screening found 100 promising keys out of 100000
Pre-screening found 100 promising keys out of 100000
Pre-screening found 100 promising keys out of 100000
Pre-screening found 100 promising keys out of 100000
Pre-screening found 100 promising keys out of 100000
Pre-screening found 100 promising keys out of 100000
Pre-screening found 100 promising keys out of 100000
Pre-screening found 100 promising keys out of 100000
Pre-screening found 100 promising keys out of 25000
Chunk B125611-T0: Screened 125000 keys, found 0 results
Pre-screening found 100 promising keys out of 25000
Pre-screenin